# LangChain Single-Agent Architecture — Lab Notes & Experiments

In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import requests
from langchain_classic.agents import create_react_agent, AgentExecutor

/tmp/ipykernel_1577778/1342358485.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY=os.getenv('GOOGLE_API_KEY')
TAVILY_API_KEY=os.getenv('TAVILY_API_KEY')
WEATHERSTACK_API_KEY=os.getenv('WEATHERSTACK_API_KEY')

In [3]:
search_tool = TavilySearchResults(max_results=1)
result = search_tool.invoke("What is capital of france")

/tmp/ipykernel_1577778/1273174157.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=1)


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=1.0,  
    max_retries=2,
)

result = model.invoke("how much is 5x5 ? ")
result.text


/home/med/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'5 x 5 = 25'

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

In [6]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [10]:
# create the agent
agent = create_react_agent(llm=model,tools=tools,prompt=prompt)

In [ ]:
# executor
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True
                               )

agent_executor

In [ ]:
# running the executor
reponse = agent_executor.invoke({"input":("find the capital of morcco"
                                          "and then finds its current weather.")})

In [84]:
print(reponse["output"])

The capital of Morocco is Rabat. The current weather in Rabat is clear with a temperature of around 72°F (feels like 74°F) and winds from the northwest at 6 mph.


In [31]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%")

In [30]:
tools = [search_tool, get_weather_data]